# rm-optpessimal-personas — Colab runner

Runs the GPU-dependent generation scripts from [puffables/rm-optpessimal-personas](https://github.com/puffables/rm-optpessimal-personas) on Colab compute.

**Before you start:**
1. `Runtime > Change runtime type > GPU > A100`. Everything here also runs on a T4/L4, but A100 cuts wall-clock time substantially — worth it if you're optimizing for speed over compute-unit cost. The two 27B Gemma-2 base models still won't fit even on a 40GB A100 (they need ~54GB just for weights); Cell 6 skips them by default regardless of GPU.
2. Create a HuggingFace token at https://huggingface.co/settings/tokens (read access is enough) and accept the license on each gated model page you need (`google/gemma-*`, `meta-llama/Llama-3.2-3B-Instruct`).
3. This repo is **private**, so cloning it needs a GitHub token too: create a fine-grained personal access token at https://github.com/settings/tokens?type=beta scoped only to the `puffables/rm-optpessimal-personas` repo, with **Contents: Read-only** permission.
4. In the Colab left sidebar, click the key icon ("Secrets") and add two secrets — `HF_TOKEN` and `GH_TOKEN` — with those tokens, toggling "Notebook access" on for both. This keeps both tokens out of the notebook itself.

**Data persistence:** the repo's `data/` folder is checked into git, so a fresh clone already has prior results, and the three checkpointing scripts (everything except `generate_base_model_logprobs.py`) will skip work that's already done. The last cell zips up anything new/changed so you can pull it back into your local clone and commit it — this notebook does not push to GitHub for you.

In [1]:
# 1. Confirm a GPU is attached
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB


In [2]:
# 2. Clone the repo (fresh each session)
# Private repo, so the clone is authenticated with GH_TOKEN (from Colab secrets).
# The token is only embedded in the URL for the clone itself; the remote is
# rewritten to drop it immediately after so it isn't left sitting in .git/config.
# Uses an absolute path so this cell is safe to re-run mid-session without
# nesting a clone inside itself.
import os
from google.colab import userdata

REPO_PATH = "puffables/rm-optpessimal-personas"
REPO_DIR = "/content/rm-optpessimal-personas"
GH_TOKEN = userdata.get("GH_TOKEN")

if not os.path.exists(REPO_DIR):
    !git clone https://{GH_TOKEN}@github.com/{REPO_PATH}.git {REPO_DIR}
%cd {REPO_DIR}
!git fetch origin kv-cached
!git checkout kv-cached
!git remote set-url origin https://github.com/{REPO_PATH}.git

Cloning into '/content/rm-optpessimal-personas'...
remote: Enumerating objects: 293, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 293 (delta 0), reused 3 (delta 0), pack-reused 276 (from 1)
Receiving objects: 100% (293/293), 193.85 MiB | 16.52 MiB/s, done.
Resolving deltas: 100% (79/79), done.
Updating files: 100% (112/112), done.
/content/rm-optpessimal-personas
From https://github.com/puffables/rm-optpessimal-personas
 * branch            kv-cached  -> FETCH_HEAD
Branch 'kv-cached' set up to track remote branch 'kv-cached' from 'origin'.
Switched to a new branch 'kv-cached'


In [3]:
# 3. Install requirements
# Colab's preinstalled torch is already CUDA-enabled, so this mainly adds
# transformers (>=4.45, for the rope_scaling fix Llama-3.x needs), accelerate,
# and the smaller deps.
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 139.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 146.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 105.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 88.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.6/180.6 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 50.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [4]:
# 4. HuggingFace auth (reads the HF_TOKEN secret set up in the sidebar)
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))

In [5]:
# 5. Sanity-check gated model access before burning compute on a script that'll 401 midway through
from huggingface_hub import model_info

for m in ["google/gemma-2b", "google/gemma-2-9b", "google/gemma-2-27b", "meta-llama/Llama-3.2-3B-Instruct"]:
    try:
        model_info(m)
        print(f"OK   {m}")
    except Exception as e:
        print(f"FAIL {m}: {e}")

OK   google/gemma-2b
OK   google/gemma-2-9b
OK   google/gemma-2-27b
OK   meta-llama/Llama-3.2-3B-Instruct


## 6. Reward model scores
By default `config/reward_models.yaml` has only one *active* (uncommented) model — `Ray2333/GRM-Llama3.2-3B-rewardmodel-ft` (3B) — the rest are commented out. The YAML's configured batch size (128) was tuned conservatively; on an A100 you can push it much higher with `--batch-size` to cut runtime.

In [ ]:
!python generate_reward_model_scores.py --batch-size 512

Skipping Ray2333/GRM-Llama3.2-3B-rewardmodel-ft — all prompts already scored
Done.


## 6b. KV-cached comparison (this branch)
The `kv-cached` branch adds a `--kv-cache` scoring path that caches the shared prompt prefix once per prompt instead of re-tokenizing and re-running the whole sequence per candidate token — much faster on a real GPU with a big batch size. Building it also surfaced two pre-existing issues in the original scoring path (duplicate BOS token, and a decode-then-re-tokenize round trip that can silently substitute a different token for a large fraction of the vocabulary), both fixed in this path by default.

This writes to a **separate output directory** — not `data/reward_model_scores` — so the checkpoint-skip logic in the script doesn't treat the prompts scored above as already done.

In [ ]:
!python generate_reward_model_scores.py --batch-size 1024 --kv-cache --output-dir data/reward_model_scores_kv_cached

Processing model: Ray2333/GRM-Llama3.2-3B-rewardmodel-ft (7 prompts to score)
Found 1 CUDA devices
Using batch size 128 with single GPU setup (cuda)
2026-08-03 13:56:19.223355: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-03 13:56:19.294912: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Loading checkpoint shards: 100% 2/2 [00:00<00:00,  3.52it/s]
Model dtype: torch.float16
Reward model Ray2333/GRM-Llama3.2-3B-rewardmodel-ft (tokenizer class PreTrainedTokenizerFast) initialized
  Prompt: greatest
  greatest: 

## 6c. Bug-fixed, no-cache comparison
Runs the same sweep through `--fixed` — the duplicate-BOS and decode/re-tokenize fixes, but *without* KV-caching (full forward pass per candidate token, same cost as the original baseline). This isolates the effect of the tokenization fix alone, separate from the caching speedup — compare this against the baseline (bug effect) and against the kv-cached run (should match closely, since kv-cached uses the same fixes; any nontrivial gap there would flag a caching bug rather than a scoring difference).

In [ ]:
!python generate_reward_model_scores.py --batch-size 1024 --fixed --output-dir data/reward_model_scores_fixed

Processing model: Ray2333/GRM-Llama3.2-3B-rewardmodel-ft (7 prompts to score)
Found 1 CUDA devices
tokenizer_config.json: 54.7kB [00:00, 79.3MB/s]
tokenizer.json: 9.09MB [00:00, 61.1MB/s]
special_tokens_map.json: 100% 434/434 [00:00<00:00, 3.87MB/s]
Using batch size 128 with single GPU setup (cuda)
config.json: 1.13kB [00:00, 6.22MB/s]
2026-08-03 12:33:50.291982: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-03 12:33:50.362217: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
model.safetensors.index.json: 21.0kB

## 6d. Diff across all three variants

In [ ]:
import pandas as pd

MODEL_CSV = "Ray2333--GRM-Llama3.2-3B-rewardmodel-ft.csv"
variants = {
    "main": pd.read_csv(f"data/reward_model_scores/{MODEL_CSV}"),
    "fixed": pd.read_csv(f"data/reward_model_scores_fixed/{MODEL_CSV}"),
    "kv_cached": pd.read_csv(f"data/reward_model_scores_kv_cached/{MODEL_CSV}"),
}

prompt_cols = [c for c in variants["main"].columns
               if c not in ("token_id", "token_name", "token_decoded")]

def compare(a, b, col):
    diff = (variants[a][col] - variants[b][col]).abs()
    top_a = set(variants[a].nlargest(10, col)["token_decoded"])
    top_b = set(variants[b].nlargest(10, col)["token_decoded"])
    overlap = len(top_a & top_b)
    return diff.mean(), diff.max(), overlap

for col in prompt_cols:
    print(f"=== {col} ===")
    for a, b, label in [("main", "fixed", "main vs fixed (bug-fix effect alone)"),
                        ("fixed", "kv_cached", "fixed vs kv_cached (caching effect alone)"),
                        ("main", "kv_cached", "main vs kv_cached (combined effect)")]:
        mean_diff, max_diff, overlap = compare(a, b, col)
        print(f"  {label:42s} mean|diff|={mean_diff:.3f}  max|diff|={max_diff:.3f}  top-10 overlap={overlap}/10")
    print()


=== greatest ===
  main vs fixed (bug-fix effect alone)       mean|diff|=0.446  max|diff|=6.219  top-10 overlap=8/10
  fixed vs kv_cached (caching effect alone)  mean|diff|=0.003  max|diff|=0.027  top-10 overlap=10/10
  main vs kv_cached (combined effect)        mean|diff|=0.446  max|diff|=6.219  top-10 overlap=8/10

=== best ===
  main vs fixed (bug-fix effect alone)       mean|diff|=0.450  max|diff|=5.945  top-10 overlap=5/10
  fixed vs kv_cached (caching effect alone)  mean|diff|=0.003  max|diff|=0.062  top-10 overlap=10/10
  main vs kv_cached (combined effect)        mean|diff|=0.450  max|diff|=5.941  top-10 overlap=5/10

=== worst ===
  main vs fixed (bug-fix effect alone)       mean|diff|=0.416  max|diff|=4.344  top-10 overlap=4/10
  fixed vs kv_cached (caching effect alone)  mean|diff|=0.004  max|diff|=0.023  top-10 overlap=10/10
  main vs kv_cached (combined effect)        mean|diff|=0.415  max|diff|=4.336  top-10 overlap=4/10

=== greatest_plain ===
  main vs fixed (bug-fix ef

## 7. Persona-conditioned reward model scores
Same active model set as above, swept over `config/personas.yaml` x `config/persona_prompts.yaml` — 227 combinations, the slowest step in this pipeline. This script's default batch size (1024) was tuned for a 97GB workstation GPU; on a 40GB A100, 384 leaves headroom for the longer persona-prefixed prompts. If you land on a smaller GPU instead, drop this back to 128 (T4) or lower.

In [8]:
!python generate_persona_reward_model_scores.py --batch-size 384

Processing model: Ray2333/GRM-Llama3.2-3B-rewardmodel-ft (245 persona/template combinations to score)
Found 1 CUDA devices
Using batch size 128 with single GPU setup (cuda)
2026-08-04 18:57:03.957115: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-04 18:57:04.024818: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Loading checkpoint shards: 100% 2/2 [00:00<00:00,  3.48it/s]
Model dtype: torch.float16
Reward model Ray2333/GRM-Llama3.2-3B-rewardmodel-ft (tokenizer class PreTrainedTokenizerFast) initialized
  great

## 7b. KV-cached persona sweep (this branch)
Same 245 (template, persona) combinations, run through `--kv-cache` — caches each prompt's prefix once instead of re-tokenizing and re-running it per candidate token, and fixes the duplicate-BOS and decode/re-tokenize round-trip issues along the way (see `experiments/tokenization_bug_findings.md` for what those bugs actually do to the results). Writes to a separate output directory so the checkpoint-skip logic doesn't treat these columns as already done.

**Batch size note**: KV-caching duplicates each cached prefix's key/value tensors across the batch dimension (`repeat_interleave`), so for the longer persona-prefixed prompts this can use *more* memory per batch item than the non-cached path at the same batch size, not less — kept at 384 to match the non-cached persona run above rather than assuming caching buys batch-size headroom too. Raise it experimentally if you have headroom on your GPU.

In [6]:
!python generate_persona_reward_model_scores.py --batch-size 384 --kv-cache --output-dir data/persona_reward_model_scores_kv_cached

Processing model: Ray2333/GRM-Llama3.2-3B-rewardmodel-ft (245 persona/template combinations to score)
Found 1 CUDA devices
tokenizer_config.json: 54.7kB [00:00, 110MB/s]
tokenizer.json: 9.09MB [00:00, 202MB/s]
special_tokens_map.json: 100% 434/434 [00:00<00:00, 3.78MB/s]
Using batch size 128 with single GPU setup (cuda)
config.json: 1.13kB [00:00, 7.62MB/s]
2026-08-04 15:23:43.319451: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-04 15:23:43.387986: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
model.safetens

## 7c. Persona-shift comparison: main vs kv-cached

In [9]:
# For each (template, persona) column scored under both pipelines, compares the persona-vs-baseline shift (Kendall's tau / RBO p=0.95 between the persona-conditioned ranking and its baseline column from persona_prompts.yaml) computed under `main` against the same shift computed under `kv_cached`. See experiments/tokenization_bug_findings.md for the methodology and the 8-persona pilot this reproduces at full scale.
import yaml
import pandas as pd
import numpy as np
from scipy.stats import kendalltau
import reassemble

# restore results data from fragmented parts

reassemble.ensure_all()

MODEL_CSV = "Ray2333--GRM-Llama3.2-3B-rewardmodel-ft.csv"
baseline_main = pd.read_csv(f"data/reward_model_scores/{MODEL_CSV}")
baseline_kv = pd.read_csv(f"data/reward_model_scores_kv_cached/{MODEL_CSV}")
persona_main = pd.read_csv(f"data/persona_reward_model_scores/{MODEL_CSV}")
persona_kv = pd.read_csv(f"data/persona_reward_model_scores_kv_cached/{MODEL_CSV}")

with open("config/persona_prompts.yaml") as f:
    templates = yaml.safe_load(f)["templates"]

def rbo(rank1, rank2, p=0.95, threshold=1e-6):
    max_depth = min(len(rank1), len(rank2), int(np.log(threshold) / np.log(p)) + 1)
    rank1, rank2 = rank1[:max_depth], rank2[:max_depth]
    weights = np.power(p, np.arange(max_depth)) * (1 - p)
    set1, set2 = set(), set()
    overlaps = np.empty(max_depth)
    for d in range(max_depth):
        set1.add(rank1[d]); set2.add(rank2[d])
        overlaps[d] = len(set1 & set2) / (d + 1)
    return float(np.dot(overlaps, weights))

def ranked_ids(df, col):
    return df.sort_values(col, ascending=False)["token_id"].values

def shift(baseline_df, persona_df, persona_col, baseline_col):
    b = baseline_df[["token_id", baseline_col]].dropna()
    p = persona_df[["token_id", persona_col]].dropna()
    m = b.merge(p, on="token_id")
    tau, _ = kendalltau(m[baseline_col], m[persona_col])
    r = rbo(ranked_ids(baseline_df, baseline_col), ranked_ids(persona_df, persona_col))
    return tau, r

rows = []
for col in persona_kv.columns:
    if "__" not in col:
        continue
    template_name = col.split("__", 1)[0]
    baseline_col = templates[template_name]["baseline_column"]
    tau_main, rbo_main = shift(baseline_main, persona_main, col, baseline_col)
    tau_kv, rbo_kv = shift(baseline_kv, persona_kv, col, baseline_col)
    rows.append({"column": col, "tau_main": tau_main, "tau_kv": tau_kv,
                 "rbo_main": rbo_main, "rbo_kv": rbo_kv})

shift_df = pd.DataFrame(rows)
shift_df["d_tau"] = shift_df["tau_kv"] - shift_df["tau_main"]
shift_df["d_rbo"] = shift_df["rbo_kv"] - shift_df["rbo_main"]
shift_df.to_csv("persona_shift_main_vs_kv_cached.csv", index=False)

print(f"n = {len(shift_df)} (template, persona) combinations")
print(f"corr(tau_main, tau_kv) across all combinations: "
      f"{shift_df['tau_main'].corr(shift_df['tau_kv']):.4f}")
print(f"corr(rbo_main, rbo_kv) across all combinations: "
      f"{shift_df['rbo_main'].corr(shift_df['rbo_kv']):.4f}")
print(f"mean d_tau: {shift_df['d_tau'].mean():+.4f}  (positive = kv-cached shows MORE shift from baseline than main)")
print(f"share of combinations with d_tau > 0: {(shift_df['d_tau'] > 0).mean():.1%}")
shift_df.sort_values("d_tau", ascending=False).head(10)


NameError: name 'sys' is not defined

## 8. Persona-conditioned base model logprobs
Uses `config/llama_base_models.yaml`, which lists only `meta-llama/Llama-3.2-3B-Instruct` — fits on a T4.

In [ ]:
!python generate_persona_base_model_logprobs.py

## 9. Base model logprobs (the heavy one)
`config/gemma_base_models.yaml` spans gemma-1/2 from 2B up to **27B**. Unlike the three scripts above, this one has no resume/skip logic — it recomputes and overwrites every model's CSV on every run, so only pick the models you actually need.

The two 27B variants need ~54GB just for weights in bf16, which doesn't fit even on Colab's 40GB A100 — the filtered config below (which drops them) is the right default regardless of GPU tier. Their outputs are already present in `data/base_model_logits/` from a prior run — only rerun them (on hardware that can actually hold them) if `config/prompts.yaml` has changed since.

In [ ]:
# Build a Colab-friendly config that excludes the 27B models (not committed to the repo)
import yaml

with open("config/gemma_base_models.yaml") as f:
    all_models = yaml.safe_load(f)

small_models = [m for m in all_models if "27b" not in m["name"]]
with open("config/gemma_base_models_no27b.yaml", "w") as f:
    yaml.safe_dump(small_models, f)

print(f"{len(small_models)}/{len(all_models)} models kept:")
for m in small_models:
    print(" ", m["name"])

In [ ]:
!python generate_base_model_logprobs.py --config gemma_base_models_no27b.yaml

# Full sweep including the 27B models — only run this on an A100 80GB:
# !python generate_base_model_logprobs.py --config gemma_base_models.yaml

## 10. Pull results back out
Zips the `data/` outputs so you can download them and merge into your local clone (`git status` there will show only the files that actually changed).

In [ ]:
from google.colab import files

!zip -qr data_outputs.zip data/reward_model_scores data/reward_model_scores_fixed data/reward_model_scores_kv_cached data/persona_reward_model_scores data/persona_reward_model_scores_kv_cached persona_shift_main_vs_kv_cached.csv data/persona_base_model_logits data/base_model_logits
files.download("data_outputs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>